In [1]:
import re
import joblib
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import (roc_auc_score, average_precision_score,
                             classification_report, confusion_matrix)

pd.set_option('display.max_columns', None)

In [2]:
df = pd.read_csv('../data/german_credit_clean.csv')

X = df.drop(columns=['default', 'sex'])
y = df['default']
sex_full = df['sex']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
sex_train = sex_full.loc[X_train.index]
sex_test = sex_full.loc[X_test.index]

print(X_train.shape, X_test.shape)

(800, 20) (200, 20)


In [3]:
categorical_cols = ['checking_account', 'credit_history', 'purpose', 'savings_account',
                    'employment_since', 'personal_status_sex', 'other_debtors', 'property',
                    'other_installment_plans', 'housing', 'job', 'telephone', 'foreign_worker']

X_train_enc = pd.get_dummies(X_train, columns=categorical_cols, drop_first=True)
X_test_enc = pd.get_dummies(X_test, columns=categorical_cols, drop_first=True)
X_train_enc, X_test_enc = X_train_enc.align(X_test_enc, join='left', axis=1, fill_value=0)
X_train_enc = X_train_enc.astype(float)
X_test_enc = X_test_enc.astype(float)

# XGBoost rejects '[', ']' and '<' in feature names
def clean_col_name(col):
    return re.sub(r'[\[\]<>]', '', col).strip()

X_train_enc.columns = [clean_col_name(c) for c in X_train_enc.columns]
X_test_enc.columns = [clean_col_name(c) for c in X_test_enc.columns]

print("Duplicate columns:", X_train_enc.columns[X_train_enc.columns.duplicated()].tolist())
print(X_train_enc.shape, X_test_enc.shape)

Duplicate columns: []
(800, 48) (200, 48)


In [4]:
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print("scale_pos_weight:", scale_pos_weight)

xgb_model = xgb.XGBClassifier(
    n_estimators=200, max_depth=4, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    eval_metric='logloss', random_state=42,
    scale_pos_weight=scale_pos_weight
)
xgb_model.fit(X_train_enc, y_train)

proba = xgb_model.predict_proba(X_test_enc)[:, 1]
pred = (proba >= 0.5).astype(int)

print("ROC-AUC:", roc_auc_score(y_test, proba))
print("PR-AUC:", average_precision_score(y_test, proba))
print("\nClassification Report:\n", classification_report(y_test, pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, pred))

scale_pos_weight: 2.3333333333333335
ROC-AUC: 0.7997619047619048
PR-AUC: 0.6648810795216648

Classification Report:
               precision    recall  f1-score   support

           0       0.83      0.80      0.81       140
           1       0.57      0.62      0.59        60

    accuracy                           0.74       200
   macro avg       0.70      0.71      0.70       200
weighted avg       0.75      0.74      0.75       200


Confusion Matrix:
 [[112  28]
 [ 23  37]]


## Hyperparameter tuning

Parameters below came from a 50-trial Optuna search (TPE sampler, 5-fold stratified CV
on the training set, ROC-AUC objective, best CV ROC-AUC = 0.8058). The search space was:
n_estimators 100-500, max_depth 2-6, learning_rate 0.01-0.2 (log), subsample 0.6-1.0,
colsample_bytree 0.6-1.0, min_child_weight 1-10, reg_alpha and reg_lambda 1e-3 to 10 (log).
The best parameters are hardcoded in the next cell so the notebook reproduces exactly
without re-running the search.

In [5]:
best_params = {
    'n_estimators': 361, 'max_depth': 6, 'learning_rate': 0.013900627200179493,
    'subsample': 0.7159209243676108, 'colsample_bytree': 0.7772773884070203,
    'min_child_weight': 2, 'reg_alpha': 0.01106905059953042,
    'reg_lambda': 0.0010004944075048096,
    'scale_pos_weight': scale_pos_weight,
    'eval_metric': 'logloss', 'random_state': 42
}

xgb_final = xgb.XGBClassifier(**best_params)
xgb_final.fit(X_train_enc, y_train)

proba_final = xgb_final.predict_proba(X_test_enc)[:, 1]
pred_final = (proba_final >= 0.5).astype(int)

print("Test ROC-AUC:", roc_auc_score(y_test, proba_final))
print("Test PR-AUC:", average_precision_score(y_test, proba_final))
print("\nClassification Report:\n", classification_report(y_test, pred_final))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, pred_final))

Test ROC-AUC: 0.8058333333333334
Test PR-AUC: 0.6604513700188

Classification Report:
               precision    recall  f1-score   support

           0       0.84      0.80      0.82       140
           1       0.58      0.63      0.60        60

    accuracy                           0.75       200
   macro avg       0.71      0.72      0.71       200
weighted avg       0.76      0.75      0.75       200


Confusion Matrix:
 [[112  28]
 [ 22  38]]


In [6]:
saved_model = joblib.load('../src/xgb_final_model.pkl')
saved_proba = saved_model.predict_proba(X_test_enc)[:, 1]
print("Predictions match saved model:", np.allclose(saved_proba, proba_final, atol=1e-6))
print("Max abs difference:", np.abs(saved_proba - proba_final).max())

for name, current in [('X_train_enc', X_train_enc), ('X_test_enc', X_test_enc)]:
    saved = pd.read_csv(f'../data/{name}.csv')
    same = list(saved.columns) == list(current.columns) and np.allclose(saved.values, current.values)
    print(f"{name} matches saved file:", same)

Predictions match saved model: True
Max abs difference: 0.0
X_train_enc matches saved file: True
X_test_enc matches saved file: True
